## Exercise — Reading what the coefficients say

### Task 1 — Change the units, change the coefficient

Start by running the setup below. It is the simulation, the train/test split, the model function and the R-squared helper from the video.

```py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

np.random.seed(42)
n = 80
mw = np.random.normal(350, 80, n)
logp = np.random.normal(2.5, 1.2, n)
tpsa = np.random.normal(80, 30, n)
heavy_atoms = mw / 14 + np.random.normal(0, 0.5, n)
log_s = 0.5 - 0.01 * mw - 0.7 * logp + 0.005 * tpsa + np.random.normal(0, 0.4, n)
df = pd.DataFrame({"mw": mw, "logp": logp, "tpsa": tpsa, "heavy_atoms": heavy_atoms, "log_s": log_s})
train = df.sample(frac=0.8, random_state=42)
test = df.drop(train.index)

def solubility_model(x, w_mw, w_logp, w_tpsa, b):
    return w_mw * x["mw"] + w_logp * x["logp"] + w_tpsa * x["tpsa"] + b

def r_squared(y_true, y_pred):
    ss_res = sum((y_true - y_pred) ** 2)
    ss_tot = sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot
```

1. Fit `solubility_model` to `train` with `curve_fit`, exactly as in the video. Unpack the result into `w_mw, w_logp, w_tpsa, b`.
2. Make a copy of `train` called `train_kda`. In the copy, express molecular weight in kDa by dividing the `"mw"` column by 1000.
3. Fit the same model to `train_kda` and unpack the result into `w_mw_kda, w_logp_kda, w_tpsa_kda, b_kda`.

Before running anything, write down two predictions:

- What happens to the MW coefficient when MW is measured in kDa?
- What happens to the logP coefficient?

Then print the MW coefficient from both fits, rounded to 4 decimals. Print the logP coefficient from both fits, rounded to 3 decimals.


### Task 2 — Fix the standardisation

A classmate wrote this loop to standardise the descriptors:

```py
train_z = train.copy()
test_z = test.copy()
for name in ["mw", "logp", "tpsa", "heavy_atoms"]:
    mean = train[name].mean()
    std = train[name].std()
    train_z[name] = (train[name] - mean) / std
    test_z[name] = (test[name] - test[name].mean()) / test[name].std()
```

The test line is wrong. It standardises the test compounds with their own mean and standard deviation, so the two tables end up on different scales and information from the test set leaks in.

Write the corrected version, in which the test set uses the training `mean` and `std`.

Then print two values, each rounded to 2 decimals:

- the mean of `train_z["mw"]`
- the mean of `test_z["mw"]`

The first should be essentially 0. The second need not be, because it is measured on the training scale.


### Task 3 — Rank the descriptors and draw the ranking

1. Fit `solubility_model` to `train_z` and unpack the result into `w_mw_z, w_logp_z, w_tpsa_z, b_z`.
2. Rank `"mw"`, `"logp"` and `"tpsa"` by the size of their coefficient, ignoring the sign. Do it as in the video: build `(abs(c), name, c)` tuples and pass them to `sorted`. Then pull out `ranked_names` and `ranked_coefs`.
3. Print `ranked_names`, from smallest to largest.
4. Draw the ranking with `plt.barh`, using the signed coefficients. Give the figure:
   - an x-label with units ("log S per standard deviation")
   - a y-label
   - a title

Before running, predict which descriptor's bar will sit at the top, and which direction it will point.


### Task 4 — Watch correlated descriptors split the credit

Here is the model from the video that puts both size descriptors in at once:

```py
def size_model(x, w_mw, w_heavy, w_logp, w_tpsa, b):
    return w_mw * x["mw"] + w_heavy * x["heavy_atoms"] + w_logp * x["logp"] + w_tpsa * x["tpsa"] + b
```

Define it. Then refit it on three random halves of `train_z`, drawn with `train_z.sample(frac=0.5, random_state=seed)` for the seeds `10`, `20` and `30`.

For each seed, print on one line, to 2 decimals:

- the MW coefficient
- the heavy-atom coefficient
- their sum

Before running, write down which of these three numbers you expect to stay roughly the same from seed to seed. Compare the sum with `w_mw_z` from Task 3.


### Task 5 — Predicts, but does not cause

Look back at the `log_s` line in the setup of Task 1: heavy atom count does not appear in it.

1. Define `heavy_model(x, w_heavy, w_logp, w_tpsa, b)`. It is `solubility_model` with `x["heavy_atoms"]` in place of `x["mw"]`.
2. Fit it to `train_z`.
3. Predict the compounds in `test_z`.

Print two test R-squared values, each rounded to 3:

- the R-squared of `heavy_model`
- the R-squared of `solubility_model`, using `w_mw_z, w_logp_z, w_tpsa_z, b_z` from Task 3

Before running, predict whether the heavy-atom model will score clearly worse, given that heavy atom count has no part in the formula that generated solubility.
